In [ ]:
import xarray as xr
import numpy as np
import xarray as xr
import pandas as pd
from sklearn.metrics import cohen_kappa_score

In [24]:
def load_landuse_nc(base_path="../../NC"):
    ds_dict = {
        "region": {
            "agri":      xr.open_dataset(f"{base_path}/region_agri_17regions.nc"),
            "forest":    xr.open_dataset(f"{base_path}/region_forest_17regions.nc"),
            "grassland": xr.open_dataset(f"{base_path}/region_grassland_17regions.nc"),
        },
        "basin": {
            "agri":      xr.open_dataset(f"{base_path}/basin_agri_17regions.nc"),
            "forest":    xr.open_dataset(f"{base_path}/basin_forest_17regions.nc"),
            "grassland": xr.open_dataset(f"{base_path}/basin_grassland_17regions.nc"),
        }
    }
    return ds_dict

In [25]:
def build_varname(region, mode, landtype):
    return f"{region}_{mode}_{landtype}"

In [26]:

def get_dominant_region_year(ds_dict,region,mode,landtypes,time):

    stacks = []

    for lt in landtypes:
        ds = ds_dict[mode][lt]
        varname = build_varname(region, mode, lt)

        da = ds[varname].sel(time=time)
        stacks.append(da)

    data = xr.concat(stacks, dim="class")
    data_filled = data.fillna(0)

    dominant = data_filled.argmax(dim="class")

    mask = data_filled.sum(dim="class") == 0
    dominant = dominant.where(~mask)

    return dominant


In [ ]:
def calc_kappa_vs_baseline(
    ds_dict,
    region,
    mode,
    landtypes,
    years,
    base_year=2005
):

    t0 = f"{base_year}-01-01"
    dom_base = get_dominant_region_year(
        ds_dict, region, mode, landtypes, t0
    )
    base = dom_base.values.flatten()

    kappa_list = []

    for year in years:
        if year == base_year:
            continue

        t = f"{year}-01-01"
        dom_cur = get_dominant_region_year(
            ds_dict, region, mode, landtypes, t
        )
        cur = dom_cur.values.flatten()

        mask = np.isfinite(base) & np.isfinite(cur)

        kappa = cohen_kappa_score(
            base[mask].astype(int),
            cur[mask].astype(int)
        )
        kappa = round(kappa, 3) 
        kappa_list.append(kappa)

    return kappa_list


In [ ]:
def calc_all_regions_kappa(
    ds_dict,
    regions,
    modes,
    landtypes,
    years,
    base_year=2005
):

    results = {}

    for reg in regions:
        print(f"Processing {reg} ...")
        results[reg] = {}

        for mode in modes:
            results[reg][mode] = calc_kappa_vs_baseline(
                ds_dict,
                reg,
                mode,
                landtypes,
                years,
                base_year
            )

    return results


In [29]:
import pandas as pd

def save_results_to_csv(
    results,
    years,
    out_csv,
    float_format="%.3f"
):
    rows = []
    years_plot = years[1:]  # 去掉基准年（2005）

    for region, mode_dict in results.items():
        for mode, kappa_list in mode_dict.items():
            for year, kappa in zip(years_plot, kappa_list):
                rows.append({
                    "region": region,
                    "mode": mode,
                    "year": year,
                    "kappa": kappa
                })

    df = pd.DataFrame(rows)

    df.to_csv(
        out_csv,
        index=False,
        float_format=float_format
    )

    return df


In [ ]:

regions = [
    "JPN","USA","CHN","IND","BRA","CAN","CIS",
    "XAF","XME","XNF","XSA","XSE","XOC",
    "XLM","XE25","XER","TUR"
]

modes = ["region", "basin"]
landtypes = ["agri", "forest", "grassland"]

years = [2005] + list(range(2010, 2101, 10))
ds_dict = load_landuse_nc("../../NC")
results = calc_all_regions_kappa(
    ds_dict,
    regions,
    modes,
    landtypes,
    years
)
df_kappa = save_results_to_csv(
    results,
    years,
    "../../CSV/kappa/kappa_by_region_year.csv"
)

Processing JPN ...
Processing USA ...
Processing CHN ...
Processing IND ...
Processing BRA ...
Processing CAN ...
Processing CIS ...
Processing XAF ...
Processing XME ...
Processing XNF ...
Processing XSA ...
Processing XSE ...
Processing XOC ...
Processing XLM ...
Processing XE25 ...
Processing XER ...
Processing TUR ...
